In [ ]:
# ============================================================
# WEEK 4 - PREDICTIVE MODELING AND OPTIMIZATION IN LOGISTICS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# ------------------------------------------------------------
# 1. LOAD CLEANED DATA
# ------------------------------------------------------------

file_path = "../data/processed/DataCoSupplyChainDataset_cleaned.csv.gz"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# ------------------------------------------------------------
# 2. CHECK TARGET VARIABLE
# ------------------------------------------------------------

target = "Late_delivery_risk"

print("\nTarget variable:")
print(df[target].value_counts())

print("\nTarget percentages:")
print(df[target].value_counts(normalize=True) * 100)

# ------------------------------------------------------------
# 3. SELECT USEFUL FEATURES
# ------------------------------------------------------------

features = [
    "Days for shipment (scheduled)",
    "Sales",
    "Order Item Discount",
    "Order Item Product Price",
    "Order Item Quantity",
    "Order Item Total",
    "Order Profit Per Order",
    "Benefit per order",
    "Sales per customer",
    "Order Item Profit Ratio",
    "Product Price"
]

# Keep only columns that actually exist
features = [col for col in features if col in df.columns]

print("\nFeatures used for prediction:")
for col in features:
    print("-", col)

X = df[features].copy()
y = df[target].copy()

# ------------------------------------------------------------
# 4. REMOVE INVALID / MISSING TARGET VALUES
# ------------------------------------------------------------

valid_rows = y.notna()

X = X.loc[valid_rows]
y = y.loc[valid_rows]

print("\nRows available for modeling:", len(X))

# ------------------------------------------------------------
# 5. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# ------------------------------------------------------------
# 6. PREPROCESSING
# ------------------------------------------------------------

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# ------------------------------------------------------------
# 7. LOGISTIC REGRESSION
# ------------------------------------------------------------

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# 8. DECISION TREE
# ------------------------------------------------------------

tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=8,
        random_state=42
    ))
])

tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)
tree_prob = tree_model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# 9. RANDOM FOREST
# ------------------------------------------------------------

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# 10. MODEL EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_model(name, y_true, predictions, probabilities):

    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions, zero_division=0)
    recall = recall_score(y_true, predictions, zero_division=0)
    f1 = f1_score(y_true, predictions, zero_division=0)
    auc = roc_auc_score(y_true, probabilities)

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")

    return {
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": auc
    }

# ------------------------------------------------------------
# 11. EVALUATE ALL MODELS
# ------------------------------------------------------------

results = []

results.append(
    evaluate_model(
        "Logistic Regression",
        y_test,
        logistic_pred,
        logistic_prob
    )
)

results.append(
    evaluate_model(
        "Decision Tree",
        y_test,
        tree_pred,
        tree_prob
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred,
        rf_prob
    )
)

results_df = pd.DataFrame(results)

print("\nMODEL COMPARISON")
print(results_df.round(4))

# ------------------------------------------------------------
# 12. SAVE MODEL RESULTS
# ------------------------------------------------------------

results_df.to_csv(
    "../visualizations/week4_model_comparison.csv",
    index=False
)

# ------------------------------------------------------------
# 13. CONFUSION MATRIX FOR RANDOM FOREST
# ------------------------------------------------------------

cm = confusion_matrix(y_test, rf_pred)

print("\nRandom Forest Confusion Matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["On Time", "Late"]
).plot(ax=ax)

ax.set_title("Random Forest - Late Delivery Prediction")

plt.tight_layout()

plt.savefig(
    "../visualizations/week4_random_forest_confusion_matrix.png",
    dpi=300
)

plt.show()

# ------------------------------------------------------------
# 14. CROSS-VALIDATION
# ------------------------------------------------------------

cv_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)

print("\nRandom Forest 5-Fold Cross-Validation F1 Scores:")
print(cv_scores)

print("Mean CV F1 Score:", round(cv_scores.mean(), 4))

# ------------------------------------------------------------
# 15. RANDOM FOREST FEATURE IMPORTANCE
# ------------------------------------------------------------

rf_classifier = rf_model.named_steps["model"]

importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_classifier.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("\nFeature Importance:")
print(importance)

importance.to_csv(
    "../visualizations/week4_feature_importance.csv",
    index=False
)

# ------------------------------------------------------------
# 16. FEATURE IMPORTANCE VISUALIZATION
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.barh(
    importance["Feature"],
    importance["Importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")

plt.gca().invert_yaxis()

plt.tight_layout()

plt.savefig(
    "../visualizations/week4_feature_importance.png",
    dpi=300
)

plt.show()

# ------------------------------------------------------------
# 17. PREDICTIVE INSIGHTS
# ------------------------------------------------------------

best_model = results_df.loc[
    results_df["F1 Score"].idxmax()
]

print("\nBEST MODEL")
print("=" * 50)
print("Model:", best_model["Model"])
print("Accuracy:", round(best_model["Accuracy"], 4))
print("Precision:", round(best_model["Precision"], 4))
print("Recall:", round(best_model["Recall"], 4))
print("F1 Score:", round(best_model["F1 Score"], 4))
print("ROC-AUC:", round(best_model["ROC-AUC"], 4))

print("\nTOP PREDICTIVE FEATURES")
print(importance.head(5))

# ------------------------------------------------------------
# 18. OPTIMIZATION RECOMMENDATIONS
# ------------------------------------------------------------

print("\nOPTIMIZATION RECOMMENDATIONS")
print("=" * 50)

print("""
1. Prioritize shipments with high predicted late-delivery risk.

2. Review scheduled shipping times and improve unrealistic delivery
   commitments.

3. Allocate additional logistics resources to shipments identified
   as high risk.

4. Use predictive risk scores to support proactive customer and
   warehouse communication.

5. Monitor the most important predictive features and use them
   in operational planning.

6. Combine predictive modeling with route, carrier, and regional
   analysis for future logistics optimization.
""")

print("\nWEEK 4 PREDICTIVE MODELING COMPLETE")